# Block 2: Local Structures

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imatge-upc/aa2/blob/main/notebooks/aa2_2_1_cnn_fundamental_layers.ipynb)

## 2.1: CNNs: Fundamental Layers & Architectures

Short PyTorch demonstrations to accompany the slides:

* **Convolution and pooling**: Inspect local operations, channels, padding and stride.
* **Receptive field**: Compare one large kernel with a stack of small kernels.
* **A small CNN**: Follow the tensor shapes from an image batch to class logits.
* **MNIST and transforms**: Load handwritten digits, resize them and run a forward pass.

The examples run on CPU. MNIST is downloaded into `./data` on the first run and reused afterwards.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(7)
torch.set_printoptions(precision=2, sci_mode=False)

### 1. Convolution: One Kernel, Many Local Windows

Images have shape **`(N, C, H, W)`**: batch size, channels, height and width. Here we use one image with one channel and a 4×4 grid of values.

[`nn.Conv2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) applies the same kernel at every position. We set its 2×2 weights to `1/4` to compute a **local average**, with no bias. These are hand-set demonstration weights; CNN training learns the weights.

With the default stride of 1 and no padding, a 2×2 kernel on a 4×4 input produces a 3×3 feature map. The first output is `(1 + 2 + 5 + 6) / 4 = 3.5`.

In [ ]:
x = torch.arange(1, 17, dtype=torch.float32).reshape(1, 1, 4, 4)
conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=2, bias=False)

with torch.no_grad():
    conv.weight.fill_(0.25)
    feature_map = conv(x)

print("Input shape:", tuple(x.shape))
print(x[0, 0])
print("\nKernel:")
print(conv.weight[0, 0].detach())
print("\nOutput shape:", tuple(feature_map.shape))
print(feature_map[0, 0])

#### Multiple Input Channels: Each Filter Uses Them All

For a standard convolution (`groups=1`, the default), **each output filter has a separate spatial kernel for every input channel**. Their contributions are summed into one output feature map. Different output filters have different weights.

Here channel 0 contains ones and channel 1 contains tens. We make two output filters, each with two 2×2 kernels. The weight tensor has shape **`(out_channels, in_channels, kH, kW)`**.

* Filter 0: kernel values 1 for channel 0 and 2 for channel 1 → `4 × 1 × 1 + 4 × 10 × 2 = 84`.
* Filter 1: kernel values 3 for channel 0 and 4 for channel 1 → `4 × 1 × 3 + 4 × 10 × 4 = 172`.

Each filter spans **both input channels**, and produces **one** output channel.

In [ ]:
multi_input = torch.cat([
    torch.ones(1, 1, 3, 3),
    10 * torch.ones(1, 1, 3, 3),
], dim=1)  # Shape: (1, 2, 3, 3)
multi_conv = nn.Conv2d(2, 2, kernel_size=2, bias=False)

with torch.no_grad():
    multi_conv.weight[0, 0].fill_(1)  # Output filter 0, input channel 0
    multi_conv.weight[0, 1].fill_(2)  # Output filter 0, input channel 1
    multi_conv.weight[1, 0].fill_(3)  # Output filter 1, input channel 0
    multi_conv.weight[1, 1].fill_(4)  # Output filter 1, input channel 1
    multi_output = multi_conv(multi_input)

print("Input shape:", tuple(multi_input.shape))
print("Weight shape:", tuple(multi_conv.weight.shape))
print("Filter 0 kernels:\n", multi_conv.weight[0].detach())
print("Filter 1 kernels:\n", multi_conv.weight[1].detach())
print("Output shape:", tuple(multi_output.shape))
print("Output feature maps:\n", multi_output[0])

contributions = (multi_input[0, :, :2, :2] *
                 multi_conv.weight[0].detach()).sum(dim=(1, 2))
print("Filter 0, first window: channel contributions =", contributions)
print("Sum =", contributions.sum().item())

#### Padding and Stride

We reuse the 4×4 input with the **same 3×3 averaging kernel** in all three cases:

* `padding=0, stride=1`: only windows fully inside the image → **2×2** output.
* `padding=1, stride=1`: add a border of zeros → **4×4** output.
* `padding=1, stride=2`: move the window two pixels at a time → **2×2** output.

Padding lets us control the spatial size; increasing stride downsamples the feature map. The averages at padded boundaries include zeros.

In [ ]:
for padding, stride in [(0, 1), (1, 1), (1, 2)]:
    conv_setting = nn.Conv2d(1, 1, kernel_size=3, padding=padding,
                             stride=stride, bias=False)
    with torch.no_grad():
        conv_setting.weight.fill_(1 / 9)
        setting_output = conv_setting(x)

    print(f"padding={padding}, stride={stride} → {tuple(setting_output.shape)}")
    print(setting_output[0, 0])
    print()

### 2. Pooling: Maximum or Average

[`nn.MaxPool2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html) keeps the largest value in each window; [`nn.AvgPool2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.AvgPool2d.html) keeps its mean. Pooling has no learnable weights.

Using 2×2 windows with stride 2 reduces our 4×4 image to 2×2. For the top-left window, max pooling gives **6** and average pooling gives **3.5**. Both preserve the batch and channel dimensions.

In [ ]:
max_pool = nn.MaxPool2d(kernel_size=2, stride=2)
avg_pool = nn.AvgPool2d(kernel_size=2, stride=2)

max_values = max_pool(x)
avg_values = avg_pool(x)

print("Max pooling:", tuple(max_values.shape))
print(max_values[0, 0])
print("\nAverage pooling:", tuple(avg_values.shape))
print(avg_values[0, 0])

### 3. Receptive Field: One 7×7 or Three 3×3 Convolutions?

The **receptive field** is the region of the original input that can influence one output position. With stride 1 and dilation 1, each additional 3×3 convolution expands its side length by 2:

**Input: 1×1 → first layer: 3×3 → second layer: 5×5 → third layer: 7×7.**

Compare standard convolutions with **64 input and output channels throughout, and no bias**. A convolution has $C_{mathrm{out}} C_{mathrm{in}} k_H k_W$ weights.

| Convolutions | Receptive field | Number of parameters |
|---|---|---|
| One 7×7 | 7×7 | $49 	imes 64^2 =$ **200,704** |
| Three 3×3 | 7×7 | $3 	imes 9 	imes 64^2 =$ **110,592** |

The stack uses about **45% fewer parameters** for the same receptive-field size. These are different computations; the stack also allows nonlinearities between convolutions.

Below, padding keeps the spatial size unchanged. The receptive-field sizes above refer to interior output positions.

In [ ]:
C = 64
wide_conv = nn.Conv2d(C, C, kernel_size=7, padding=3, bias=False)
stacked_convs = nn.Sequential(
    nn.Conv2d(C, C, kernel_size=3, padding=1, bias=False),
    nn.Conv2d(C, C, kernel_size=3, padding=1, bias=False),
    nn.Conv2d(C, C, kernel_size=3, padding=1, bias=False),
)

print(f"One 7×7:    {sum(p.numel() for p in wide_conv.parameters()):,} parameters")
print(f"Three 3×3:  {sum(p.numel() for p in stacked_convs.parameters()):,} parameters")

receptive_field = 1
for depth, conv_layer in enumerate(stacked_convs, start=1):
    receptive_field += conv_layer.kernel_size[0] - 1  # Stride = dilation = 1
    print(f"After layer {depth}: {receptive_field}×{receptive_field} receptive field")

### 4. A Complete Small CNN

This **LeNet-style** network uses ReLU and max pooling. It illustrates the architecture rather than reproducing historical LeNet-5 exactly.

In `Conv2d(1, 6, 5)`, `1` is the number of input channels, `6` is the number of output feature maps and `5` is the kernel size. The second convolution combines the six input channels to produce sixteen feature maps.

[`nn.Flatten`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Flatten.html) preserves the batch dimension and turns each `16 × 5 × 5` feature tensor into a vector of **400** values. The linear layers produce **10 logits**, one score per digit class.

In [ ]:
model = nn.Sequential(                  # Input: N × 1 × 32 × 32
    nn.Conv2d(1, 6, kernel_size=5),     # N × 6 × 28 × 28
    nn.ReLU(),
    nn.MaxPool2d(2),                   # N × 6 × 14 × 14

    nn.Conv2d(6, 16, kernel_size=5),    # N × 16 × 10 × 10
    nn.ReLU(),
    nn.MaxPool2d(2),                   # N × 16 × 5 × 5

    nn.Flatten(),                      # N × 400
    nn.Linear(400, 120), nn.ReLU(),
    nn.Linear(120, 84), nn.ReLU(),
    nn.Linear(84, 10),                 # N × 10 logits
)

x_batch = torch.randn(8, 1, 32, 32)
with torch.inference_mode():
    logits = model(x_batch)

print("Model", model)
print("Input shape:", tuple(x_batch.shape))
print("Logits shape:", tuple(logits.shape))
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

We can follow the shape after each layer with a simple loop:

In [ ]:
with torch.inference_mode():
    h = x_batch
    for layer in model:
        h = layer(h)
        print(f"{layer.__class__.__name__:12s} → {tuple(h.shape)}")

### 5. MNIST: Dataset → Transforms → Mini-Batch

A torchvision [`dataset`](https://docs.pytorch.org/vision/stable/datasets.html) returns an image and its label. [`transforms`](https://docs.pytorch.org/vision/stable/transforms.html) preprocess each image; a `DataLoader` groups samples into mini-batches.

We use [`MNIST`](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.MNIST.html), whose labels are digits 0–9. `Resize` changes its 28×28 grayscale images to 32×32 to match our CNN. `ToTensor` converts each PIL image to a float tensor of shape `(1, 32, 32)`, with pixel values in `[0, 1]`.

In [ ]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
])

dataset = datasets.MNIST(root="./data", train=False, download=True,
                         transform=transform)
loader = DataLoader(dataset, batch_size=8, shuffle=False)
images, labels = next(iter(loader))

print("Dataset size:", len(dataset))
print("Images:", tuple(images.shape))
print("Labels:", labels.tolist())

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, image, label in zip(axes, images, labels):
    ax.imshow(image[0], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"Label: {label.item()}")
    ax.axis("off")
plt.tight_layout()
plt.show()

The same CNN accepts this real image batch. `model.eval()` selects evaluation mode; `torch.inference_mode()` disables gradient tracking for the forward pass.

The weights are randomly initialized, so the scores do not yet represent a learned digit classifier. The output contains raw logits, as expected by `CrossEntropyLoss`.

In [ ]:
model.eval()
with torch.inference_mode():
    logits = model(images)

print("MNIST input:", tuple(images.shape))
print("Class logits:", tuple(logits.shape))
print("First image logits:", logits[0])

### 6. Torchvision Models: Inspect VGG16

[`torchvision.models.vgg16`](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.vgg16.html) provides a ready-made architecture. Its `features` stack 3×3 convolutions, ReLU activations and max pooling; its `classifier` contains fully connected layers.

We use `weights=None` to initialize the model randomly and inspect its architecture without downloading pretrained weights or running inference.

In [ ]:
from torchvision.models import vgg16

vgg = vgg16(weights=None)
print(vgg)

### PyTorch Layer Reference

The same layer families work with one, two or three spatial dimensions:

| Operation | 1D: `(N, C, L)` | 2D: `(N, C, H, W)` | 3D: `(N, C, D, H, W)` |
|---|---|---|---|
| Convolution | [`nn.Conv1d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv1d.html) | [`nn.Conv2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) | [`nn.Conv3d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv3d.html) |
| Max pooling | [`nn.MaxPool1d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool1d.html) | [`nn.MaxPool2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html) | [`nn.MaxPool3d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool3d.html) |
| Average pooling | [`nn.AvgPool1d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.AvgPool1d.html) | [`nn.AvgPool2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.AvgPool2d.html) | [`nn.AvgPool3d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.AvgPool3d.html) |

[`torch.flatten(x, start_dim=1)`](https://docs.pytorch.org/docs/stable/generated/torch.flatten.html) is the function equivalent of `nn.Flatten()` used above. Other pooling options include [`nn.AdaptiveAvgPool2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.AdaptiveAvgPool2d.html), which lets us specify the output spatial size.